# 🚀 Como Usar o NVS Benchmark

Este notebook é um guia prático para usar a ferramenta de benchmarking de síntese de novas perspectivas.

## Pré-requisitos

- Python 3.10+
- Ambiente virtual ativado
- Dependências instaladas (`pip install -e .`)
- Dataset Blender (opcional, será baixado automaticamente)

## 1. Explorando Métodos Disponíveis

In [ ]:
import subprocess
import json
from pathlib import Path

# Listar todos os métodos disponíveis
result = subprocess.run(
    ["python", "-m", "nvs_benchmark.cli", "list-methods", "--detailed"],
    capture_output=True,
    text=True
)
print(result.stdout)

### Entendendo os Métodos

- **NeRF Static**: Renderização neural com campos de radiância estáticos
  - Rápido em CPU
  - Ótimo para cenas estáticas
  - PSNR esperado: ~31 dB

- **NeRF Dynamic (D-NeRF)**: Extensão temporal do NeRF
  - Suporta cenas com movimento
  - Mais lento que NeRF estático
  - PSNR esperado: ~30 dB

- **Gaussian Splatting (3DGS)**: Renderização com splatting gaussiano
  - Muito rápido (com GPU)
  - Qualidade excelente
  - Requer CUDA
  - PSNR esperado: ~33 dB

- **4D Gaussian Splatting**: Extensão dinâmica do 3DGS
  - Cenas dinâmicas em alta velocidade
  - Requer GPU e dataset específico
  - PSNR esperado: ~32 dB

## 2. Explorando Datasets Disponíveis

In [ ]:
# Listar datasets e verificar quais estão disponíveis
result = subprocess.run(
    ["python", "-m", "nvs_benchmark.cli", "list-datasets", "--check-files"],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print("Warnings/Errors:")
    print(result.stderr)

### Datasets Disponíveis

- **blender_synthetic**: Cenas 3D renderizadas sinteticamente
  - Objetos: Lego, Chair, Drums, Ficus, Hotdog, Materials, Mic, Ship
  - Ótimo para testes iniciais
  - Tamanho: ~500MB

- **d_nerf**: Cenas dinâmicas (movimento temporal)
  - Bouncing Balls, Jumping Jacks, Hook, etc.
  - Para testar métodos dinâmicos
  - Tamanho: ~1GB

- **custom**: Seus próprios datasets
  - Requer arquivo `transforms_train.json`

## 3. Entendendo Presets e Iterações

In [ ]:
# Ver todos os presets disponíveis
result = subprocess.run(
    ["python", "-m", "nvs_benchmark.cli", "presets-list"],
    capture_output=True,
    text=True
)
print(result.stdout)

### Explicação dos Presets

| Preset | Iterações | Tempo | Use | 
|--------|-----------|-------|-----|
| **smoke** | 100 | 1-2 min | Teste rápido do pipeline |
| **quick** | 1,000 | 5-15 min | Protótipo, teste inicial |
| **preview** | 10,000 | 30-60 min | Demo, visualização |
| **standard** | 50,000 | 2-6h | Comparação com papers |
| **full** | 200,000 | 12-48h | Reprodução exata |

**Dica**: Comece com `--preset quick` para validar setup antes de rodar presets maiores.

## 4. Estimando Tempo de Execução

In [ ]:
# Estimar tempo para diferentes configurações
configs = [
    ("nerf_static", "quick"),
    ("nerf_static", "standard"),
    ("nerf_dynamic", "quick"),
    ("gs_static", "quick"),  # Pode ser mais lento
]

print("⏱️  Estimativas de Tempo por Configuração\n")
print(f"{'Método':<20} {'Preset':<15} {'Tempo Estimado'}")
print("="*60)

for method, preset in configs:
    result = subprocess.run(
        ["python", "-m", "nvs_benchmark.cli", "estimate-time",
         "--method", method, "--preset", preset],
        capture_output=True,
        text=True
    )
    # Extrair apenas a linha da estimativa
    lines = result.stdout.strip().split("\n")
    for line in lines:
        if "horas" in line or "minutos" in line:
            print(f"{method:<20} {preset:<15} {line}")

## 5. Validando Configuração (Dry-Run)

In [ ]:
# Validar se podemos rodar um benchmark
result = subprocess.run(
    ["python", "-m", "nvs_benchmark.cli", "validate-run",
     "--method", "nerf_static",
     "--dataset", "blender_synthetic",
     "--root", "./data/blender_synthetic/nerf_synthetic/lego",
     "--preset", "quick"],
    capture_output=True,
    text=True
)
print(result.stdout)

## 6. Executando um Benchmark

In [ ]:
# Executar um benchmark com método NeRF e preset quick
import time

print("🚀 Iniciando benchmark com NeRF Static + Preset Quick")
print("Tempo estimado: 5-15 minutos\n")

start_time = time.time()

result = subprocess.run(
    ["python", "-m", "nvs_benchmark.cli", "method-run",
     "--method", "nerf_static",
     "--dataset", "blender_synthetic",
     "--root", "./data/blender_synthetic/nerf_synthetic/lego",
     "--preset", "quick",
     "--compute-metrics",
     "--output-dir", "./artifacts/tutorial",
     "--snapshot-file", "./artifacts/tutorial/results.json"],
    capture_output=True,
    text=True
)

elapsed = time.time() - start_time

print(result.stdout)
if result.stderr:
    print("\nErrors/Warnings:")
    print(result.stderr)

print(f"\n⏱️  Tempo total: {elapsed/60:.1f} minutos")

## 7. Carregando e Analisando Resultados

In [ ]:
# Carregar métricas do snapshot
snapshot_path = Path("./artifacts/tutorial/results.json")

if snapshot_path.exists():
    with open(snapshot_path) as f:
        metrics = json.load(f)
    
    print("📊 Métricas Obtidas:\n")
    print(json.dumps(metrics, indent=2))
else:
    print(f"⚠️  Arquivo de métricas não encontrado em {snapshot_path}")
    print("Execute um benchmark primeiro com --compute-metrics")

## 8. Entendendo as Métricas

In [ ]:
# Explicar cada métrica
metrics_explanation = {
    "PSNR": {
        "nome": "Peak Signal-to-Noise Ratio",
        "unidade": "dB (decibéis)",
        "interpretacao": "Quanto maior, melhor a qualidade. >30 dB é muito bom.",
        "range": "0-60 dB",
        "baseline_nerf": 31.01,
        "baseline_gs": 33.32,
    },
    "SSIM": {
        "nome": "Structural Similarity",
        "unidade": "0-1",
        "interpretacao": "Quanto mais próximo de 1, melhor. Avalia similaridade estrutural.",
        "range": "0-1",
        "baseline_nerf": 0.947,
        "baseline_gs": 0.968,
    },
    "LPIPS": {
        "nome": "Learned Perceptual Image Patch Similarity",
        "unidade": "0-10",
        "interpretacao": "Quanto menor, melhor. Avalia percepção visual humana. <0.1 é excelente.",
        "range": "0-10",
        "baseline_nerf": 0.082,
        "baseline_gs": 0.043,
    },
    "FPS": {
        "nome": "Frames Per Second",
        "unidade": "fps",
        "interpretacao": "Taxa de renderização. Maior = mais rápido. >30 fps é tempo real.",
        "range": "0-1000+",
    },
}

print("📐 Explicação das Métricas de Qualidade\n")
for metric_name, info in metrics_explanation.items():
    print(f"**{metric_name}**: {info['nome']}")
    print(f"  Unidade: {info['unidade']}")
    print(f"  Range: {info['range']}")
    print(f"  Interpretação: {info['interpretacao']}")
    if "baseline_nerf" in info:
        print(f"  Baseline NeRF: {info['baseline_nerf']}")
    if "baseline_gs" in info:
        print(f"  Baseline 3DGS: {info['baseline_gs']}")
    print()

## 9. Comparando Múltiplos Métodos

In [ ]:
# Rodar múltiplos métodos para comparação
import pandas as pd
import os

methods = ["nerf_static", "nerf_dynamic"]
results = []

print("🔄 Executando benchmarks para comparação...\n")

for method in methods:
    print(f"Processando {method}...")
    
    result = subprocess.run(
        ["python", "-m", "nvs_benchmark.cli", "method-run",
         "--method", method,
         "--dataset", "blender_synthetic",
         "--root", "./data/blender_synthetic/nerf_synthetic/lego",
         "--preset", "quick",
         "--compute-metrics",
         "--output-dir", "./artifacts/comparison",
         "--append-snapshot",
         "--snapshot-file", "./artifacts/comparison/all_methods.json"],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"  ✓ {method} concluído")
    else:
        print(f"  ✗ {method} falhou")

print("\n✓ Todos os benchmarks concluídos!")

# Carregar resultados compilados
snapshot_path = Path("./artifacts/comparison/all_methods.json")
if snapshot_path.exists():
    with open(snapshot_path) as f:
        all_results = json.load(f)
    
    # Converter para DataFrame para visualização melhor
    df = pd.DataFrame(all_results).T if isinstance(all_results, dict) else pd.DataFrame(all_results)
    print("\n📊 Resultados Compilados:\n")
    print(df[["method", "psnr", "ssim", "lpips", "fps", "train_seconds", "inference_seconds"]])

## 10. Gerando Relatórios

In [ ]:
# Gerar relatório HTML automaticamente
result = subprocess.run(
    ["python", "-m", "nvs_benchmark.cli", "report-generate",
     "--snapshot-file", "./artifacts/comparison/all_methods.json",
     "--output-dir", "./artifacts/comparison/reports",
     "--report-name", "comparison_report",
     "--no-pdf"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Relatório gerado com sucesso!")
    print(result.stdout)
else:
    print("✗ Erro ao gerar relatório:")
    print(result.stderr)

## 🎯 Próximos Passos

1. **Testar com Diferentes Cenas**
   - Use outras cenas do Blender (Chair, Drums, Ficus, etc.)
   - Veja como os métodos se comportam em diferentes tipos de objetos

2. **Escalar para Presets Maiores**
   - Comece com `--preset quick`
   - Avance para `--preset standard` após validar setup
   - Use `--preset full` para reprodução exata

3. **Testar Métodos com GPU** (Opcional)
   - 3D Gaussian Splatting é muito mais rápido com GPU
   - Se tiver CUDA disponível, compare com NeRF

4. **Analisar Resultados Visualmente**
   - Use `ui-preview` para visualizar renders
   - Compare imagens lado-a-lado

5. **Customizar Datasets**
   - Use seu próprio dataset com formato COLMAP ou transforms.json
   - Veja documentação em docs/IMPLEMENTATION_DETAILS.md

## 📚 Recursos Adicionais

- [GETTING_STARTED_PRELIMINARY_RESULTS.md](../GETTING_STARTED_PRELIMINARY_RESULTS.md) - Guia para primeiros resultados
- [docs/IMPLEMENTATION_DETAILS.md](../docs/IMPLEMENTATION_DETAILS.md) - Detalhes técnicos
- Papers referenciados em `configs/paper_baselines.json`